# NYC Airbnb Price Category Classifier

This notebook classifies NYC Airbnb listings as **high-priced** or **low-priced** (`price_category`) from listing attributes. It walks through exploratory data analysis, preprocessing, a tuned logistic regression baseline, and a neural network, then compares the two models.

**Label coding:** `price_category` is encoded alphabetically, so `high` = 0 and `low` = 1. The `f1_score(..., average='binary')` values below therefore describe the **low-priced (majority) class**. The per-class reports at the end show both classes.

**Contents**
1. Setup and data loading
2. Exploratory data analysis
3. Ethical considerations
4. Data preparation
5. Logistic regression (baseline)
6. Neural network
7. Model comparison

## 1. Setup and data loading

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow info logs
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from tensorflow import keras

In [ ]:
# Look for the data next to the repo (local run), then fall back to GitHub (e.g. Colab).
LOCAL_PATHS = ["../data/airbnbListingsData.csv.gz", "data/airbnbListingsData.csv.gz"]
REMOTE_URL = ("https://raw.githubusercontent.com/NehaM2025/airbnb-price-category-predictor/"
              "main/data/airbnbListingsData.csv.gz")

airbnb_path = next((p for p in LOCAL_PATHS if os.path.exists(p)), REMOTE_URL)
df = pd.read_csv(airbnb_path)
print("Loaded from:", airbnb_path)
df.head()

## 2. Exploratory data analysis

### Class distribution

In [ ]:
class_dis = df["price_category"].value_counts()

plt.figure(figsize=(5, 5))
sns.countplot(data=df, x="price_category", order=class_dis.index)
plt.title("Distribution of Price Categories")
plt.xlabel("Price Category")
plt.ylabel("# of Listings")
plt.show()

print(class_dis)
print("\nMajority-class baseline accuracy (always predict 'low'): %.4f" % (class_dis.max() / class_dis.sum()))

### Dataset overview and missing values

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(15)

### Price distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x=df["price"])
plt.title("Distribution of Listing Prices")
plt.xlabel("Price")
plt.show()

### Price category by listing attributes

In [ ]:
plt.figure(figsize=(5, 5))
plt.ylim(0, 200)
sns.countplot(data=df, x="availability_365", hue="price_category")
plt.title("Price Category by Availability 365")
plt.xlabel("Availability 365")
plt.ylabel("# of Listings")
plt.show()

In [ ]:
plt.figure(figsize=(5, 5))
sns.countplot(data=df, x="room_type", hue="price_category")
plt.title("Price Category by Room Type")
plt.xlabel("Room Type")
plt.ylabel("# of Listings")
plt.xticks(rotation=20)
plt.show()

In [ ]:
plt.figure(figsize=(5, 5))
sns.countplot(data=df, x="bedrooms", hue="price_category")
plt.title("Price Category by # of Bedrooms")
plt.xlabel("# of Bedrooms")
plt.ylabel("# of Listings")
plt.show()

In [ ]:
plt.figure(figsize=(5, 5))
sns.countplot(data=df, x="bathrooms", hue="price_category")
plt.title("Price Category by # of Bathrooms")
plt.xlabel("# of Bathrooms")
plt.ylabel("# of Listings")
plt.show()

In [ ]:
plt.figure(figsize=(5, 5))
sns.countplot(data=df, x="host_is_superhost", hue="price_category")
plt.title("Price Category by Superhost Status")
plt.xlabel("Superhost Status")
plt.ylabel("# of Listings")
plt.show()

### Correlation with the label

In [ ]:
# Encode the label numerically (alphabetical: high = 0, low = 1)
df["price_category_num"] = df["price_category"].astype("category").cat.codes
print(dict(enumerate(df["price_category"].astype("category").cat.categories)))

corr = df.select_dtypes(include="number").corr()["price_category_num"].sort_values(ascending=False)
print(corr)

### EDA summary

**Missing values.** Several features have many missing values. `host_response_rate`, `host_acceptance_rate`, `host_about`, and `neighborhood_overview` are each missing for roughly 9,000 or more listings, and they are among the features least correlated with the label, so they are left out of the feature set. `bedrooms` and `beds` have fewer missing values and are strongly correlated with the label, so they are kept and their missing values are filled with the median.

**Feature and label relationships.** The label is built from the listing price, so `price` itself must not be used as a feature. Apart from price, `accommodates`, `bedrooms`, `beds`, and `bathrooms` show the strongest correlations with the label, which suggests that larger properties are more likely to be high-priced. Most other numeric variables are only weakly correlated with it, and no single feature predicts the label on its own.

**Class imbalance.** Low-priced listings outnumber high-priced ones roughly 3 to 1. To keep the model from defaulting to the majority class, the train/test split is stratified on the label and logistic regression uses `class_weight="balanced"`.

**Non-numeric values.** Categorical variables such as room type, borough, and host location must be encoded numerically before modeling.

## 3. Ethical considerations

**Scope.** The data contains only NYC listings from one platform at one point in time. The findings cannot be generalized to other cities, to hosts who are not on this platform, or necessarily to today's market, because prices in New York are higher than in many other places and the market changes over time.

**Proxy features.** Features such as neighborhood or borough can act as proxies for income, socioeconomic status, or race. A model that relies heavily on them can reinforce existing pricing patterns and put hosts in some neighborhoods at a disadvantage by overestimating or underestimating the value of their listings.

**Harm to hosts.** Hosts who use the predictions to help set prices can lose money or bookings if the model misclassifies their listing. The impact varies with how much a host relies on the model: hosts who have spent less time on pricing are likely to depend on it more than experienced hosts.

**Recommendation.** Use the predictions as one input among several, never as the only basis for a pricing decision.

## 4. Data preparation

### Drop and impute missing values

In [ ]:
# Drop columns with a very large number of missing values (about 9,800 to 11,800 each)
cols_to_drop = [
    "host_about",
    "neighborhood_overview",
    "host_response_rate",
    "host_acceptance_rate",
]
df = df.drop(columns=cols_to_drop)

# Median imputation for numeric columns (robust to outliers)
for col in ["bedrooms", "beds", "bathrooms"]:
    df[col] = df[col].fillna(df[col].median())

# Fill missing categorical values with "Unknown" so no false category is introduced
df["host_location"] = df["host_location"].fillna("Unknown")

df.isnull().sum().sort_values(ascending=False).head(5)

### Build the feature matrix

- `price` and the label columns are removed: the label is created from price, so keeping it would let the model cheat.
- Free-text and (mostly) unique-per-row columns are removed: one-hot encoding them is very expensive and adds little signal.
- Remaining categorical columns are one-hot encoded so that every input is numeric.

In [ ]:
y = df["price_category_num"]

X = df.drop(columns=["price_category", "price_category_num", "price"], errors="ignore")
X = X.drop(columns=["host_name", "amenities", "name", "description"], errors="ignore")

categorical_cols = X.select_dtypes(include=["object", "string"]).columns
print("Categorical columns:", list(categorical_cols))

X = pd.get_dummies(X, columns=categorical_cols, drop_first=True, dtype="int8")

print("Shape after encoding:", X.shape)
print("Missing values remaining:", int(X.isnull().sum().sum()))
print(X.dtypes.value_counts())
X.head()

## 5. Logistic regression (baseline)

The data is split 90% train and 10% test, stratified on the label so both sets keep the same class balance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=1234
)
print("Train:", X_train.shape, "Test:", X_test.shape)

### Untuned model

Define a small helper that prints a per-class report, so both classes are visible and not only the majority class.

In [ ]:
CLASS_NAMES = ["high", "low"]  # label codes 0 and 1

def report(name, y_true, y_pred):
    print("=" * 60)
    print(name)
    print("=" * 60)
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))
    print("Confusion matrix (rows = actual, columns = predicted):")
    print(pd.DataFrame(confusion_matrix(y_true, y_pred),
                       index=["actual " + c for c in CLASS_NAMES],
                       columns=["pred " + c for c in CLASS_NAMES]))
    print()

In [ ]:
model = LogisticRegression(class_weight="balanced", max_iter=10000, random_state=1234)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score (low class):", f1_score(y_test, y_pred, average="binary"))

### Hyperparameter tuning

`GridSearchCV` searches the regularization strength `C` with 5-fold cross-validation. This step can take several minutes.

In [ ]:
cs = [10 ** i for i in range(-5, 5)]
param_grid = {"C": cs, "max_iter": [10000]}

print("Running grid search...")
grid = GridSearchCV(model, param_grid, cv=5, n_jobs=-1)
grid_search = grid.fit(X_train, y_train)
print("Done")

print("Best parameters:", grid.best_params_)
print("Best cross-validation accuracy:", grid.best_score_)
best_c = grid_search.best_params_["C"]

### Final logistic regression model

In [ ]:
final_model = LogisticRegression(C=best_c, class_weight="balanced", max_iter=3000, random_state=1234)
final_model.fit(X_train, y_train)
y_pred_lr = final_model.predict(X_test)

logreg_accuracy = accuracy_score(y_test, y_pred_lr)
logreg_f1 = f1_score(y_test, y_pred_lr, average="binary")

print("Accuracy:", logreg_accuracy)
print("F1 Score (low class):", logreg_f1)
report("Logistic regression (tuned)", y_test, y_pred_lr)

### Feature coefficients

Larger absolute coefficients mean a stronger influence on the prediction. The features here are not scaled, so coefficient sizes also depend on each feature's units. Treat them as a rough guide rather than an exact ranking.

In [ ]:
coef = pd.DataFrame({"Feature": X.columns, "Coefficient": final_model.coef_[0]})
coef["Absolute Value"] = coef["Coefficient"].abs()
coef = coef.sort_values("Absolute Value", ascending=False)
coef.head(15)

## 6. Neural network

Inputs are standardized (fit on the training data only), because neural networks train poorly on features with very different scales.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
n_features = X_train_scaled.shape[1]

### Architecture

- **Two hidden layers (64 then 32 neurons, ReLU):** deep enough to learn more complex patterns than a single layer, but small enough to train quickly. The shrinking width gradually condenses the information before the prediction.
- **One sigmoid output neuron:** this is a binary classification problem, so the output is the probability of class 1 (`low`).

In [ ]:
nn_model = keras.Sequential()
nn_model.add(keras.layers.Input(shape=(n_features,)))
nn_model.add(keras.layers.Dense(64, activation="relu"))
nn_model.add(keras.layers.Dense(32, activation="relu"))
nn_model.add(keras.layers.Dense(1, activation="sigmoid"))
nn_model.summary()

### Compile and train

A learning rate of 0.001 makes gradual weight updates, since large updates could overshoot a good solution. Training runs for 30 epochs as a balance between learning time and training cost, with 20% of the training data held out for validation.

In [ ]:
sgd_optimizer = keras.optimizers.SGD(learning_rate=0.001)
loss_fn = keras.losses.BinaryCrossentropy(from_logits=False)
nn_model.compile(optimizer=sgd_optimizer, loss=loss_fn, metrics=["accuracy"])


class ProgBarLoggerNEpochs(keras.callbacks.Callback):
    """Print the training metrics every `every_n` epochs."""

    def __init__(self, num_epochs: int, every_n: int = 50):
        super().__init__()
        self.num_epochs = num_epochs
        self.every_n = every_n

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every_n == 0:
            s = "Epoch [{}/ {}]".format(epoch + 1, self.num_epochs)
            logs_s = ["{}: {:.4f}".format(k.capitalize(), v) for k, v in logs.items()]
            print(", ".join([s] + logs_s))

In [ ]:
t0 = time.time()

num_epochs = 30
history = nn_model.fit(
    X_train_scaled, y_train,
    epochs=num_epochs,
    verbose=0,
    callbacks=[ProgBarLoggerNEpochs(num_epochs, every_n=5)],
    validation_split=0.2,
)

print("Elapsed time: %.2fs" % (time.time() - t0))

### Training curves

In [ ]:
epochs = range(1, num_epochs + 1)

plt.plot(epochs, history.history["loss"], label="Training Loss")
plt.plot(epochs, history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.plot(epochs, history.history["accuracy"], label="Training Accuracy")
plt.plot(epochs, history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

A growing gap between the training and validation curves means the network is starting to overfit. Early stopping (see the next steps in the README) would address this.

### Evaluate on the test set

In [ ]:
nn_probabilities = nn_model.predict(X_test_scaled, verbose=0)
nn_predictions = (nn_probabilities >= 0.5).astype(int).flatten()

nn_accuracy = accuracy_score(y_test, nn_predictions)
nn_f1 = f1_score(y_test, nn_predictions)

print("Neural Network Accuracy:", nn_accuracy)
print("Neural Network F1 Score (low class):", nn_f1)
report("Neural network", y_test, nn_predictions)

## 7. Model comparison

In [ ]:
baseline_acc = (y_test == 1).mean()  # always predict 'low'

comparison = pd.DataFrame({
    "Metric": ["Accuracy", "F1 (low class)", "F1 (high class)", "Macro F1"],
    "Always-predict-low baseline": [
        baseline_acc,
        f1_score(y_test, np.ones_like(y_test)),
        0.0,
        f1_score(y_test, np.ones_like(y_test), average="macro"),
    ],
    "Logistic Regression": [
        logreg_accuracy, logreg_f1,
        f1_score(y_test, y_pred_lr, pos_label=0),
        f1_score(y_test, y_pred_lr, average="macro"),
    ],
    "Neural Network": [
        nn_accuracy, nn_f1,
        f1_score(y_test, nn_predictions, pos_label=0),
        f1_score(y_test, nn_predictions, average="macro"),
    ],
})
print(comparison.round(4).to_string(index=False))

### Discussion

**Reading the table.** About 74% of listings are low-priced, so a model that always answers "low" already reaches that accuracy. A useful model has to beat this baseline and also score well on the high-priced class, which is the harder and more valuable one for hosts. Compare the models using accuracy, the high-class F1, and the macro F1 together, not accuracy alone.

**Complexity versus performance.** The neural network is harder to design, tune, and interpret than logistic regression. It is justified only if it clearly improves the high-class and macro F1 scores over logistic regression, and the improvement is large enough to matter to the client.

**Which model to use.** If predictive performance matters most and the model is retrained only when the market changes, the neural network is the better choice when it scores higher. Training is a one-time cost rather than a recurring one. If each host must be told which attributes drive their classification, logistic regression is the better choice because its coefficients can be explained directly.

**What I would do next**
- Search more hyperparameter combinations (depth, neurons per layer, learning rate, optimizer) instead of a single configuration.
- Add early stopping so weights stop updating once validation loss stops improving, which reduces overfitting.
- Spend more time on feature engineering: prune weak one-hot features, scale skewed numeric features, and analyze feature importance. Many one-hot columns (for example `host_location`) add training time without adding much signal.
- Standardize the inputs for logistic regression as well, so the two models are compared on equal terms and the coefficients are comparable.